In [1]:
import os

WEIGHTS_DIR = '/kaggle/input/datasets/sooryakiranv2120/pre-trained-weights'

print("Checking pretrained weights...")
expected = ['efficientnet_b4.pth', 'vit_base_patch16_224.pth', 'convnext_base.pth']

all_found = True
for fname in expected:
    fpath = os.path.join(WEIGHTS_DIR, fname)
    if os.path.exists(fpath):
        size_mb = os.path.getsize(fpath) / (1024*1024)
        print(f"   {fname} ({size_mb:.1f} MB)")
    else:
        print(f"   {fname} NOT FOUND")
        all_found = False

if all_found:
    print("\n All weights found")
else:
    print("\n Some weights missing — fix before running!")
    print(f"Available files in {WEIGHTS_DIR}:")
    if os.path.exists(WEIGHTS_DIR):
        for f in os.listdir(WEIGHTS_DIR):
            print(f"  {f}")
    else:
        print(f"  Directory not found: {WEIGHTS_DIR}")
        print("  Check your dataset name in the Input panel")

Checking pretrained weights...
   efficientnet_b4.pth (74.5 MB)
   vit_base_patch16_224.pth (330.3 MB)
   convnext_base.pth (338.1 MB)

 All weights found


In [2]:
!pip install timm albumentations pywavelets -q

import os, zipfile, warnings, pickle
import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from PIL import Image
from tqdm import tqdm
from scipy.fftpack import dct
from scipy.optimize import minimize
import pywt
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts

warnings.filterwarnings('ignore')
torch.backends.cudnn.benchmark = True
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {DEVICE}")
if DEVICE == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

WEIGHTS_DIR = '/kaggle/input/datasets/sooryakiranv2120/pre-trained-weights'

Device: cuda
GPU: Tesla T4


In [3]:
BASE_DIR   = "/kaggle/input/datasets/sooryakiranv2120/etsy-image-classification"
IMAGES_DIR = os.path.join(BASE_DIR, "genai_image_challenge", "images_final_sample")

print(f"Images available: {len(os.listdir(IMAGES_DIR))}")

train_df = pd.read_csv(os.path.join(BASE_DIR, "train.csv"))
test_df  = pd.read_csv(os.path.join(BASE_DIR, "test.csv"))

label_map = {'authentic': 0, 'ai_generated': 1}
if train_df['ground_truth'].dtype == object:
    train_df['label'] = train_df['ground_truth'].map(label_map)
else:
    train_df['label'] = train_df['ground_truth'].astype(int)

def find_path(image_id):
    for ext in ['', '.jpg', '.jpeg', '.png', '.webp']:
        p = os.path.join(IMAGES_DIR, f'{image_id}{ext}')
        if os.path.exists(p): return p
    return None

train_df['path'] = train_df['image_id'].apply(find_path)
test_df['path']  = test_df['image_id'].apply(find_path)
train_df = train_df.dropna(subset=['path']).reset_index(drop=True)

print(f"Train: {len(train_df)}  |  Test: {len(test_df)}")
print(train_df['label'].value_counts())

Images available: 6858
Train: 4800  |  Test: 2058
label
0    2485
1    2315
Name: count, dtype: int64


In [4]:
import os
import numpy as np
os.makedirs('/kaggle/working/models', exist_ok=True)

def load_if_exists(prefix):
    oof  = f'/kaggle/working/models/{prefix}_oof_probs.npy'
    test = f'/kaggle/working/models/{prefix}_test_probs.npy'
    if os.path.exists(oof) and os.path.exists(test):
        print(f" {prefix} loaded from checkpoint")
        return np.load(test), np.load(oof), True
    print(f" {prefix} not found — will train")
    return None, None, False

eff_test_probs,      eff_oof_probs,      eff_done      = load_if_exists('efficientnet_b4')
vit_test_probs,      vit_oof_probs,      vit_done      = load_if_exists('vit_base')
convnext_test_probs, convnext_oof_probs, convnext_done = load_if_exists('convnext_base')

freq_done = os.path.exists('/kaggle/working/models/freq_train_probs.npy')
if freq_done:
    freq_train_probs = np.load('/kaggle/working/models/freq_train_probs.npy')
    freq_test_probs  = np.load('/kaggle/working/models/freq_test_probs.npy')
    print(" freq loaded from checkpoint")
else:
    print(" freq not found — will train")

 efficientnet_b4 not found — will train
 vit_base not found — will train
 convnext_base not found — will train
 freq not found — will train


In [5]:
CFG = {
    'img_size_eff': 300,
    'img_size_vit': 224,
    'eff_model': 'efficientnet_b4',
    'vit_model': 'vit_base_patch16_224',
    'eff_epochs': 20,
    'vit_epochs': 28,
    'eff_bs': 16,
    'vit_bs': 12,
    'lr': 2e-4,
    'lr_backbone': 3e-6,
    'unfreeze_epoch': 6,
    'n_folds': 5,
    'seed': 42,
    'label_smooth': 0.03,
    'focal_gamma': 2.0,
    'mixup_alpha': 0.2,
    'mixup_prob': 0.5,
    'tta_steps': 10,
    'num_workers': 4,
    'device': DEVICE,
}

torch.manual_seed(CFG['seed'])
np.random.seed(CFG['seed'])
os.makedirs('/kaggle/working/models', exist_ok=True)
print("Config ready.")

Config ready.


In [6]:
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

def make_transforms(sz):
    train_tfm = A.Compose([
        A.Resize(sz, sz),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.2),
        A.RandomRotate90(p=0.3),
        A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=15, p=0.4),
        A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.03, p=0.5),
        A.CLAHE(p=0.3),
        A.GaussNoise(var_limit=(5,20), p=0.3),
        A.CoarseDropout(max_holes=4, max_height=32, max_width=32, p=0.2),
        A.Normalize(mean=MEAN, std=STD),
        ToTensorV2(),
    ])
    val_tfm = A.Compose([A.Resize(sz, sz), A.Normalize(mean=MEAN, std=STD), ToTensorV2()])
    tta_tfm = A.Compose([
        A.Resize(sz, sz),
        A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5), A.RandomRotate90(p=0.5),
        A.Normalize(mean=MEAN, std=STD), ToTensorV2(),
    ])
    return train_tfm, val_tfm, tta_tfm


class ImgDataset(Dataset):
    def __init__(self, df, transform, has_labels=True):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.has_labels = has_labels

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = cv2.imread(row['path'])
        if img is None:
            img = np.array(Image.open(row['path']).convert('RGB'))
        else:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = self.transform(image=img)['image']
        if self.has_labels:
            return img, torch.tensor(row['label'], dtype=torch.long)
        return img


class AIDetector(nn.Module):
    def __init__(self, model_name, pretrained=True, drop=0.3):
        super().__init__()
        self.backbone = timm.create_model(model_name, pretrained=False, num_classes=0)
        wpath = os.path.join(WEIGHTS_DIR, f'{model_name}.pth')
        if pretrained and os.path.exists(wpath):
            state = torch.load(wpath, map_location='cpu')
            state = {k: v for k, v in state.items()
                     if not k.startswith('head') and not k.startswith('classifier')}
            self.backbone.load_state_dict(state, strict=False)
            print(f"   Loaded {model_name} from local weights")
        else:
            print(f"   Local weights not found, downloading {model_name}...")
            self.backbone = timm.create_model(model_name, pretrained=True, num_classes=0)
        feat = self.backbone.num_features
        self.head = nn.Sequential(
            nn.LayerNorm(feat), nn.Dropout(drop),
            nn.Linear(feat, 512), nn.GELU(), nn.Dropout(drop/2),
            nn.Linear(512, 2),
        )

    def freeze_backbone(self):
        for p in self.backbone.parameters(): p.requires_grad = False

    def unfreeze_backbone(self):
        for p in self.backbone.parameters(): p.requires_grad = True

    def forward(self, x):
        return self.head(self.backbone(x))


class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, smoothing=0.03, weight=None):
        super().__init__()
        self.gamma     = gamma
        self.smoothing = smoothing
        self.weight    = weight

    def forward(self, pred, target):
        n       = pred.size(1)
        log_p   = F.log_softmax(pred, dim=1)
        smooth  = self.smoothing / (n - 1)
        one_hot = torch.zeros_like(log_p).scatter_(1, target.unsqueeze(1), 1)
        soft    = one_hot*(1-self.smoothing) + (1-one_hot)*smooth
        ce      = -(soft * log_p).sum(dim=1)
        pt      = torch.exp(-ce)
        focal   = (1 - pt)**self.gamma * ce
        if self.weight is not None:
            focal = focal * self.weight[target]
        return focal.mean()


def rand_bbox(size, lam):
    W, H = size[2], size[3]
    cut_rat = np.sqrt(1. - lam)
    cut_w, cut_h = int(W * cut_rat), int(H * cut_rat)
    cx, cy = np.random.randint(W), np.random.randint(H)
    bbx1 = np.clip(cx - cut_w//2, 0, W)
    bby1 = np.clip(cy - cut_h//2, 0, H)
    bbx2 = np.clip(cx + cut_w//2, 0, W)
    bby2 = np.clip(cy + cut_h//2, 0, H)
    return bbx1, bby1, bbx2, bby2


def mixup_cutmix(imgs, labels, alpha=0.2, prob=0.5):
    if np.random.rand() > prob:
        return imgs, labels
    idx = torch.randperm(imgs.size(0)).to(imgs.device)
    if np.random.rand() < 0.5:
        lam = np.random.beta(alpha, alpha)
        mixed = lam * imgs + (1 - lam) * imgs[idx]
        return mixed, (labels, labels[idx], lam)
    else:
        lam = np.random.beta(1.0, 1.0)
        bbx1, bby1, bbx2, bby2 = rand_bbox(imgs.size(), lam)
        imgs[:, :, bbx1:bbx2, bby1:bby2] = imgs[idx, :, bbx1:bbx2, bby1:bby2]
        lam = 1 - ((bbx2-bbx1)*(bby2-bby1)/(imgs.size(-1)*imgs.size(-2)))
        return imgs, (labels, labels[idx], lam)


def mixup_loss(criterion, pred, label_info):
    if isinstance(label_info, tuple):
        la, lb, lam = label_info
        return lam * criterion(pred, la) + (1-lam) * criterion(pred, lb)
    return criterion(pred, label_info)


def train_one_epoch(model, loader, opt, criterion, device, scaler):
    model.train()
    total = 0.0
    for imgs, labels in tqdm(loader, leave=False, desc='  train'):
        imgs, labels = imgs.to(device), labels.to(device)
        imgs, label_info = mixup_cutmix(imgs, labels, CFG['mixup_alpha'], CFG['mixup_prob'])
        opt.zero_grad()
        with torch.cuda.amp.autocast():
            loss = mixup_loss(criterion, model(imgs), label_info)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt)
        scaler.update()
        total += loss.item()
    return total / len(loader)


@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()
    total, preds, labs = 0.0, [], []
    for imgs, labels in tqdm(loader, leave=False, desc='  val'):
        imgs, labels = imgs.to(device), labels.to(device)
        logits = model(imgs)
        total += criterion(logits, labels).item()
        preds.extend(logits.argmax(1).cpu().numpy())
        labs.extend(labels.cpu().numpy())
    return total/len(loader), f1_score(labs, preds)


@torch.no_grad()
def get_probs_tta(model, df, device, sz, steps):
    model.eval()
    _, _, tta_tfm = make_transforms(sz)
    all_probs = []
    for _ in range(steps):
        loader = DataLoader(ImgDataset(df, tta_tfm, has_labels=False),
                            batch_size=32, shuffle=False, num_workers=2)
        batch_probs = []
        for imgs in loader:
            p = F.softmax(model(imgs.to(device)), dim=1).cpu().numpy()
            batch_probs.append(p)
        all_probs.append(np.concatenate(batch_probs))
    return np.mean(all_probs, axis=0)


print("All utilities ready.")

All utilities ready.


In [7]:
def train_model_kfold(model_name, img_size, epochs, bs, save_prefix, train_df, test_df):
    train_tfm, val_tfm, _ = make_transforms(img_size)
    skf = StratifiedKFold(n_splits=CFG['n_folds'], shuffle=True, random_state=CFG['seed'])

    fold_scores   = []
    test_probs_all = []
    oof_probs     = np.zeros((len(train_df), 2))
    fold_models   = []

    for fold, (tr_idx, vl_idx) in enumerate(skf.split(train_df, train_df['label'])):
        print(f"\n{'='*60}\n{model_name} — Fold {fold+1}/{CFG['n_folds']}\n{'='*60}")

        tr_df = train_df.iloc[tr_idx].reset_index(drop=True)
        vl_df = train_df.iloc[vl_idx].reset_index(drop=True)

        tr_loader = DataLoader(ImgDataset(tr_df, train_tfm), batch_size=bs,
                               shuffle=True, num_workers=CFG['num_workers'], pin_memory=True)
        vl_loader = DataLoader(ImgDataset(vl_df, val_tfm), batch_size=bs,
                               shuffle=False, num_workers=CFG['num_workers'], pin_memory=True)

        model = AIDetector(model_name).to(DEVICE)
        model.freeze_backbone()

        counts = tr_df['label'].value_counts().sort_index().values
        w = torch.tensor([len(tr_df)/(2*c) for c in counts], dtype=torch.float).to(DEVICE)
        criterion = FocalLoss(gamma=CFG['focal_gamma'],
                              smoothing=CFG['label_smooth'], weight=w)

        opt = optim.AdamW(model.head.parameters(), lr=CFG['lr'], weight_decay=1e-4)
        sch = CosineAnnealingWarmRestarts(opt, T_0=CFG['unfreeze_epoch'], eta_min=1e-7)
        scaler = torch.cuda.amp.GradScaler()

        best_f1 = 0.0
        ckpt = f'/kaggle/working/models/{save_prefix}_fold{fold}.pth'

        for ep in range(1, epochs+1):
            if ep == CFG['unfreeze_epoch']:
                model.unfreeze_backbone()
                opt = optim.AdamW([
                    {'params': model.backbone.parameters(), 'lr': CFG['lr_backbone']},
                    {'params': model.head.parameters(), 'lr': CFG['lr']},
                ], weight_decay=1e-4)
                sch = CosineAnnealingWarmRestarts(opt, T_0=epochs-ep+1, eta_min=1e-7)
                print(f"  [Ep {ep}] Backbone unfrozen.")

            tr_loss = train_one_epoch(model, tr_loader, opt, criterion, DEVICE, scaler)
            vl_loss, vl_f1 = validate(model, vl_loader, criterion, DEVICE)
            sch.step()

            star = ''
            if vl_f1 > best_f1:
                best_f1 = vl_f1
                torch.save(model.state_dict(), ckpt)
                star = ' '
            print(f"  Ep {ep:02d}/{epochs} | TrL {tr_loss:.4f} | VlL {vl_loss:.4f} | F1 {vl_f1:.4f}{star}")

        print(f"\n  Best fold F1: {best_f1:.4f}")
        fold_scores.append(best_f1)

        model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
        model.eval()
        fold_models.append(model)

        vl_loader2 = DataLoader(ImgDataset(vl_df, val_tfm), batch_size=bs,
                                shuffle=False, num_workers=2)
        fold_oof = []
        with torch.no_grad():
            for imgs, _ in vl_loader2:
                p = F.softmax(model(imgs.to(DEVICE)), dim=1).cpu().numpy()
                fold_oof.append(p)
        oof_probs[vl_idx] = np.concatenate(fold_oof)

        print(f"  Running TTA on test set...")
        tp = get_probs_tta(model, test_df, DEVICE, img_size, CFG['tta_steps'])
        test_probs_all.append(tp)

        np.save(f'/kaggle/working/models/{save_prefix}_oof_probs.npy', oof_probs)
        np.save(f'/kaggle/working/models/{save_prefix}_test_probs.npy',
                np.mean(test_probs_all, axis=0))
        print(f"  Saved checkpoint probs (fold {fold+1} complete).")

    oof_f1 = f1_score(train_df['label'], oof_probs.argmax(1))
    print(f"\n{'='*60}")
    print(f"{model_name} RESULTS:")
    print(f"  Per-fold F1: {[f'{s:.4f}' for s in fold_scores]}")
    print(f"  Mean: {np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f}")
    print(f"  OOF F1: {oof_f1:.4f}")

    return np.mean(test_probs_all, axis=0), oof_probs, oof_f1, fold_models


if not eff_done:
    print("Starting EfficientNet-B4 training (5-fold)...")
    eff_test_probs, eff_oof_probs, eff_oof_f1, eff_fold_models = train_model_kfold(
        model_name=CFG['eff_model'],
        img_size=CFG['img_size_eff'],
        epochs=CFG['eff_epochs'],
        bs=CFG['eff_bs'],
        save_prefix='efficientnet_b4',
        train_df=train_df,
        test_df=test_df,
    )
    print(f"\nEfficientNet OOF F1: {eff_oof_f1:.4f}")
else:
    eff_oof_f1 = f1_score(train_df['label'], eff_oof_probs.argmax(1))
    eff_fold_models = []
    print(f" Skipping EfficientNet — loaded from checkpoint (F1: {eff_oof_f1:.4f})")

Starting EfficientNet-B4 training (5-fold)...

efficientnet_b4 — Fold 1/5
   Loaded efficientnet_b4 from local weights


  Ep 01/20 | TrL 0.1810 | VlL 0.1454 | F1 0.7235 


  Ep 02/20 | TrL 0.1573 | VlL 0.1467 | F1 0.7439 


  Ep 03/20 | TrL 0.1500 | VlL 0.1430 | F1 0.7243


  Ep 04/20 | TrL 0.1451 | VlL 0.1442 | F1 0.7527 


...

  Ep 16/20 | TrL 0.1189 | VlL 0.1358 | F1 0.7740 


  Ep 17/20 | TrL 0.1117 | VlL 0.1311 | F1 0.7838 


  Ep 18/20 | TrL 0.1132 | VlL 0.1363 | F1 0.7712


  Ep 19/20 | TrL 0.1111 | VlL 0.1329 | F1 0.7735


  Ep 20/20 | TrL 0.1140 | VlL 0.1348 | F1 0.7689

  Best fold F1: 0.7838
  Running TTA on test set...
  Saved checkpoint probs (fold 5 complete).

efficientnet_b4 RESULTS:
  Per-fold F1: ['0.7946', '0.7984', '0.7855', '0.7800', '0.7838']
  Mean: 0.7885 ± 0.0069
  OOF F1: 0.7884

EfficientNet OOF F1: 0.7884


In [8]:
if not vit_done:
    print("Starting ViT-B/16 training (5-fold)...")
    vit_test_probs, vit_oof_probs, vit_oof_f1, vit_fold_models = train_model_kfold(
        model_name=CFG['vit_model'],
        img_size=CFG['img_size_vit'],
        epochs=CFG['vit_epochs'],
        bs=CFG['vit_bs'],
        save_prefix='vit_base',
        train_df=train_df,
        test_df=test_df,
    )
    print(f"\nViT OOF F1: {vit_oof_f1:.4f}")
else:
    vit_oof_f1 = f1_score(train_df['label'], vit_oof_probs.argmax(1))
    vit_fold_models = []
    print(f" Skipping ViT — loaded from checkpoint (F1: {vit_oof_f1:.4f})")

Starting ViT-B/16 training (5-fold)...

vit_base_patch16_224 — Fold 1/5
   Loaded vit_base_patch16_224 from local weights


  Ep 01/28 | TrL 0.1760 | VlL 0.1484 | F1 0.7144 


  Ep 02/28 | TrL 0.1540 | VlL 0.1407 | F1 0.7393 


  Ep 03/28 | TrL 0.1502 | VlL 0.1384 | F1 0.7577 


  Ep 04/28 | TrL 0.1436 | VlL 0.1325 | F1 0.7739 


...

  Ep 24/28 | TrL 0.0626 | VlL 0.1082 | F1 0.8767


  Ep 25/28 | TrL 0.0677 | VlL 0.1106 | F1 0.8792 


  Ep 26/28 | TrL 0.0648 | VlL 0.1101 | F1 0.8780


  Ep 27/28 | TrL 0.0676 | VlL 0.1081 | F1 0.8791


  Ep 28/28 | TrL 0.0622 | VlL 0.1102 | F1 0.8805 

  Best fold F1: 0.8805
  Running TTA on test set...
  Saved checkpoint probs (fold 5 complete).

vit_base_patch16_224 RESULTS:
  Per-fold F1: ['0.8830', '0.8905', '0.8938', '0.8786', '0.8805']
  Mean: 0.8853 ± 0.0059
  OOF F1: 0.8853

ViT OOF F1: 0.8853


In [9]:
if not convnext_done:
    print("Starting ConvNeXt-Base training (5-fold)...")
    convnext_test_probs, convnext_oof_probs, convnext_oof_f1, convnext_fold_models = train_model_kfold(
        model_name='convnext_base',
        img_size=288,
        epochs=22,
        bs=14,
        save_prefix='convnext_base',
        train_df=train_df,
        test_df=test_df,
    )
    print(f"\nConvNeXt OOF F1: {convnext_oof_f1:.4f}")
else:
    convnext_oof_f1 = f1_score(train_df['label'], convnext_oof_probs.argmax(1))
    convnext_fold_models = []
    print(f" Skipping ConvNeXt — loaded from checkpoint (F1: {convnext_oof_f1:.4f})")

Starting ConvNeXt-Base training (5-fold)...

convnext_base — Fold 1/5
   Loaded convnext_base from local weights


  Ep 01/22 | TrL 0.1725 | VlL 0.1377 | F1 0.7639 


  Ep 02/22 | TrL 0.1515 | VlL 0.1276 | F1 0.7817 


  Ep 03/22 | TrL 0.1405 | VlL 0.1275 | F1 0.7800


  Ep 04/22 | TrL 0.1335 | VlL 0.1259 | F1 0.7922 


...

  Ep 18/22 | TrL 0.0808 | VlL 0.0956 | F1 0.8568


  Ep 19/22 | TrL 0.0781 | VlL 0.0968 | F1 0.8563


  Ep 20/22 | TrL 0.0807 | VlL 0.0975 | F1 0.8554


  Ep 21/22 | TrL 0.0742 | VlL 0.0971 | F1 0.8551


  Ep 22/22 | TrL 0.0722 | VlL 0.0964 | F1 0.8548

  Best fold F1: 0.8613
  Running TTA on test set...
  Saved checkpoint probs (fold 5 complete).

convnext_base RESULTS:
  Per-fold F1: ['0.8747', '0.8845', '0.8868', '0.8850', '0.8613']
  Mean: 0.8785 ± 0.0096
  OOF F1: 0.8785

ConvNeXt OOF F1: 0.8785


In [10]:
if not freq_done:

    def extract_freq_features(path, size=128):
        try:
            img  = cv2.imread(path)
            if img is None: raise ValueError
            img  = cv2.resize(img, (size, size))
            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY).astype(np.float32)
            feats = []

            # FFT radial spectrum
            fft = np.fft.fft2(gray)
            mag = np.log1p(np.abs(np.fft.fftshift(fft)))
            cy, cx = size//2, size//2
            yg, xg = np.ogrid[:size, :size]
            r = np.sqrt((xg-cx)**2+(yg-cy)**2).astype(int)
            radial = [mag[r==i].mean() if (r==i).any() else 0.0
                      for i in range(r.max()+1)]
            idx = np.linspace(0, len(radial)-1, 32).astype(int)
            feats.extend([radial[i] for i in idx])

            # DCT
            d  = dct(dct(gray, axis=0, norm='ortho'), axis=1, norm='ortho')
            hf = d[size//2:, size//2:].flatten()
            feats.extend([hf.mean(), hf.std(), np.percentile(hf,25),
                          np.percentile(hf,75), np.abs(hf).mean()])

            # Noise residual
            noise = gray - cv2.GaussianBlur(gray, (5,5), 1.5)
            feats.extend([noise.mean(), noise.std(), np.percentile(noise,5),
                          np.percentile(noise,95), np.abs(noise).mean()])

            # Wavelet
            coeffs = pywt.wavedec2(gray, 'db4', level=2)
            for lvl in coeffs[1:]:
                for detail in lvl:
                    feats.extend([detail.mean(), detail.std(),
                                   np.abs(detail).mean()])

            # Per-channel colour
            for ch in range(3):
                c = img[:,:,ch].astype(np.float32).flatten()
                feats.extend([c.mean(), c.std(),
                              np.percentile(c,25), np.percentile(c,75)])

            # JPEG block artefact
            bv = np.array([gray[by:by+8, bx:bx+8].var()
                           for by in range(0, size-8, 8)
                           for bx in range(0, size-8, 8)])
            feats.extend([bv.mean(), bv.std(),
                          np.percentile(bv,10), np.percentile(bv,90)])

            # Chrominance noise
            ycrcb = cv2.cvtColor(img, cv2.COLOR_BGR2YCrCb).astype(np.float32)
            for ch in [1, 2]:
                n = ycrcb[:,:,ch] - cv2.GaussianBlur(ycrcb[:,:,ch], (5,5), 1.5)
                feats.extend([n.var(), np.abs(n).mean()])

            # LBP histogram
            def lbp_hist(g, nb=8):
                lbp = np.zeros_like(g, dtype=np.uint8)
                for dy, dx in [(-1,-1),(-1,0),(-1,1),(0,1),
                               (1,1),(1,0),(1,-1),(0,-1)]:
                    s = np.roll(np.roll(g, dy, 0), dx, 1)
                    lbp = (lbp << 1) | (g >= s).astype(np.uint8)
                h, _ = np.histogram(lbp[1:-1,1:-1], bins=nb, range=(0,255))
                return (h / (h.sum() + 1e-8)).tolist()
            feats.extend(lbp_hist(gray.astype(np.uint8)))

            return np.array(feats, dtype=np.float32)
        except Exception:
            return np.zeros(118, dtype=np.float32)

    print("Extracting frequency features (train)...")
    X_freq_train = np.array([extract_freq_features(p) for p in tqdm(train_df['path'])])
    y_freq_train = train_df['label'].values

    print("Extracting frequency features (test)...")
    X_freq_test = np.array([extract_freq_features(p) for p in tqdm(test_df['path'])])

    print("Training frequency GBM...")
    freq_clf = Pipeline([
        ('scaler', StandardScaler()),
        ('clf', GradientBoostingClassifier(n_estimators=300, max_depth=5,
                                            learning_rate=0.03, subsample=0.8,
                                            min_samples_leaf=20, random_state=42))
    ])
    freq_clf.fit(X_freq_train, y_freq_train)

    freq_train_probs = freq_clf.predict_proba(X_freq_train)
    freq_test_probs  = freq_clf.predict_proba(X_freq_test)

    np.save('/kaggle/working/models/freq_train_probs.npy', freq_train_probs)
    np.save('/kaggle/working/models/freq_test_probs.npy',  freq_test_probs)

    freq_f1 = f1_score(y_freq_train, freq_train_probs.argmax(1))
    print(f"Frequency GBM train F1: {freq_f1:.4f}")

else:
    print(" Skipping freq — loaded from checkpoint")
    freq_f1 = f1_score(train_df['label'], freq_train_probs.argmax(1))
    print(f"Freq F1 (checkpoint): {freq_f1:.4f}")

Extracting frequency features (train)...


100%|██████████| 4800/4800 [02:02<00:00, 39.31it/s]


Extracting frequency features (test)...


100%|██████████| 2058/2058 [00:48<00:00, 42.37it/s]


Training frequency GBM...
Frequency GBM train F1: 0.9145


In [11]:
from sklearn.linear_model import LogisticRegression

def temperature_scale(probs, T):
    logits = np.log(probs + 1e-10)
    logits /= T
    exp = np.exp(logits - logits.max(axis=1, keepdims=True))
    return exp / exp.sum(axis=1, keepdims=True)

true_labels = train_df['label'].values

best_T = {}
for name, probs in [('eff', eff_oof_probs), ('vit', vit_oof_probs),
                     ('convnext', convnext_oof_probs), ('freq', freq_train_probs)]:
    best_t, best_nll = 1.0, np.inf
    for T in np.arange(0.4, 2.2, 0.1):
        cal = temperature_scale(probs, T)
        nll = -np.mean(np.log(cal[np.arange(len(true_labels)), true_labels] + 1e-10))
        if nll < best_nll:
            best_nll, best_t = nll, T
    best_T[name] = best_t
    print(f"{name} best temperature: {best_t:.1f}")

eff_oof_cal       = temperature_scale(eff_oof_probs,       best_T['eff'])
vit_oof_cal       = temperature_scale(vit_oof_probs,       best_T['vit'])
convnext_oof_cal  = temperature_scale(convnext_oof_probs,  best_T['convnext'])
freq_oof_cal      = temperature_scale(freq_train_probs,    best_T['freq'])

eff_test_cal      = temperature_scale(eff_test_probs,      best_T['eff'])
vit_test_cal      = temperature_scale(vit_test_probs,      best_T['vit'])
convnext_test_cal = temperature_scale(convnext_test_probs, best_T['convnext'])
freq_test_cal     = temperature_scale(freq_test_probs,     best_T['freq'])

print("Calibration done.")

eff best temperature: 0.5
vit best temperature: 0.8
convnext best temperature: 0.5
freq best temperature: 0.4
Calibration done.


In [12]:
from scipy.optimize import minimize

# ── Grid search ───────────────────────────────────────────────────────────
print("Grid search weights...")
oof_stack  = [eff_oof_cal,  vit_oof_cal,  convnext_oof_cal,  freq_oof_cal]
test_stack = [eff_test_cal, vit_test_cal, convnext_test_cal, freq_test_cal]

best_f1_grid = 0.0
best_we, best_wv, best_wc, best_wf = 0.25, 0.30, 0.25, 0.20

for we in np.arange(0.15, 0.55, 0.05):
    for wv in np.arange(0.25, 0.60, 0.05):
        for wc in np.arange(0.10, 0.40, 0.05):
            wf = round(1.0 - we - wv - wc, 3)
            if wf < 0.0 or wf > 0.25:
                continue
            ens = (we*eff_oof_cal + wv*vit_oof_cal +
                   wc*convnext_oof_cal + wf*freq_oof_cal)
            f1 = f1_score(true_labels, ens.argmax(1))
            if f1 > best_f1_grid:
                best_f1_grid = f1
                best_we, best_wv, best_wc, best_wf = we, wv, wc, wf

best_w = np.array([best_we, best_wv, best_wc, best_wf])
print(f"Best weights → Eff:{best_we:.2f} ViT:{best_wv:.2f} ConvNeXt:{best_wc:.2f} Freq:{best_wf:.2f}")
print(f"Best OOF ensemble F1: {best_f1_grid:.4f}")

# ── Threshold search ──────────────────────────────────────────────────────
oof_ens = sum(wi*pi for wi, pi in zip(best_w, oof_stack))
best_thresh, best_f1_thresh = 0.5, 0.0
for t in np.arange(0.28, 0.72, 0.002):
    preds = (oof_ens[:, 1] > t).astype(int)
    f1 = f1_score(true_labels, preds)
    if f1 > best_f1_thresh:
        best_f1_thresh, best_thresh = f1, t

print(f"Optimal threshold: {best_thresh:.3f}  (default was 0.500)")
print(f"F1 with threshold: {best_f1_thresh:.4f}")


Grid search weights...
Best weights → Eff:0.15 ViT:0.40 ConvNeXt:0.20 Freq:0.25
Best OOF ensemble F1: 0.9250
Optimal threshold: 0.576  (default was 0.500)
F1 with threshold: 0.9270


In [13]:
def get_probs_at_scale(models, df, sz, bs=16, tta_rounds=5):
    tfm = A.Compose([
        A.Resize(sz, sz),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        A.Normalize(mean=MEAN, std=STD),
        ToTensorV2(),
    ])
    all_fold_probs = []
    for model in models:
        model.eval()
        fold_tta = []
        for _ in range(tta_rounds):
            loader = DataLoader(ImgDataset(df, tfm, has_labels=False),
                                batch_size=bs, shuffle=False, num_workers=2)
            batch_probs = []
            with torch.no_grad():
                for imgs in loader:
                    p = F.softmax(model(imgs.to(DEVICE)), dim=1).cpu().numpy()
                    batch_probs.append(p)
            fold_tta.append(np.concatenate(batch_probs))
        all_fold_probs.append(np.mean(fold_tta, axis=0))
    return np.mean(all_fold_probs, axis=0)

if not eff_fold_models or not vit_fold_models or not convnext_fold_models:
    print(" Fold models not in memory — using single-scale probs")
    eff_test_ms  = eff_test_cal
    vit_test_ms  = vit_test_cal
    cnx_test_ms  = convnext_test_cal
else:
    print("Multi-scale TTA — EfficientNet at 256px...")
    eff_test_256 = get_probs_at_scale(eff_fold_models, test_df, sz=256, bs=16, tta_rounds=5)

    print("Multi-scale TTA — ViT at 224px (more rounds)...")
    vit_test_224 = get_probs_at_scale(vit_fold_models, test_df, sz=224, bs=12, tta_rounds=8)

    print("Multi-scale TTA — ConvNeXt at 256px...")
    cnx_test_256 = get_probs_at_scale(convnext_fold_models, test_df, sz=256, bs=16, tta_rounds=5)

    eff_test_ms  = 0.5 * eff_test_cal  + 0.5 * temperature_scale(eff_test_256,  best_T['eff'])
    vit_test_ms  = 0.4 * vit_test_cal  + 0.6 * temperature_scale(vit_test_224,  best_T['vit'])
    cnx_test_ms  = 0.5 * convnext_test_cal + 0.5 * temperature_scale(cnx_test_256, best_T['convnext'])
    print("Multi-scale TTA done.")

Multi-scale TTA — EfficientNet at 256px...
Multi-scale TTA — ViT at 224px (more rounds)...
Multi-scale TTA — ConvNeXt at 256px...
Multi-scale TTA done.


In [14]:
test_stack_ms    = [eff_test_ms, vit_test_ms, cnx_test_ms, freq_test_cal]
final_test_probs = sum(wi*pi for wi, pi in zip(best_w, test_stack_ms))
final_preds      = (final_test_probs[:, 1] > best_thresh).astype(int)

submission = test_df[['image_id']].copy()
submission['prediction'] = final_preds
submission.to_csv('/kaggle/working/submission.csv', index=False)

print(f"Submission saved!")
print(f"Threshold used   : {best_thresh:.3f}")
print(f"Expected OOF F1  : {best_f1_thresh:.4f}")
print(f"Prediction distribution:\n{submission['prediction'].value_counts()}")

default_preds = (final_test_probs[:, 1] > 0.5).astype(int)
print(f"\nWith threshold 0.5:    {np.sum(default_preds==1)} AI-generated")
print(f"With threshold {best_thresh:.3f}: {np.sum(final_preds==1)} AI-generated")

Submission saved!
Threshold used   : 0.576
Expected OOF F1  : 0.9270
Prediction distribution:
prediction
0    1119
1     939
Name: count, dtype: int64

With threshold 0.5:    1034 AI-generated
With threshold 0.576: 939 AI-generated
